### STT

In [20]:
# ============== STT =======================
import speech_recognition as sr


def speech_to_text(phrase_time_limit: int = None):
    speech = sr.Recognizer()
    print("🎤 Listening Please speak...")

    with sr.Microphone() as source:

        try:
            speech.adjust_for_ambient_noise(source)
            audio = speech.listen(source, phrase_time_limit=phrase_time_limit)
            user_input = speech.recognize_google(audio)
            return f"[User]: {user_input}"
        except sr.UnknownValueError:
            return None  # Return empty string if speech is unintelligible
        except sr.RequestError as e:
            print(f"Could not request results from Google Speech Recognition service; {e}")
            return None
        except sr.WaitTimeoutError:
            return None
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            return None

In [39]:
# ============= New STT =====================
import speech_recognition as sr

def speech_to_text(phrase_time_limit: int = None):
    recognizer = sr.Recognizer()

    # 🎛️ Tuning for better sensitivity
    recognizer.energy_threshold = 300  # Lower = more sensitive to quiet sounds
    recognizer.dynamic_energy_threshold = False  # Use fixed threshold
    recognizer.pause_threshold = 0.8  # Allow natural pauses in speech

    print("🎤 Listening... Please speak clearly.")

    with sr.Microphone() as source:
        try:
            # 🧠 Calibrate for ambient noise (longer duration = better accuracy)
            recognizer.adjust_for_ambient_noise(source, duration=1)

            # 🎙️ Listen with time limit
            audio = recognizer.listen(source, phrase_time_limit=phrase_time_limit)

            # 🗣️ Convert speech to text
            user_input = recognizer.recognize_google(audio)
            return f"[User]: {user_input}"

        except sr.UnknownValueError:
            return ""  # No understandable speech
        except sr.RequestError as e:
            print(f"❌ API error: {e}")
            return ""
        except sr.WaitTimeoutError:
            return ""
        except Exception as e:
            print(f"⚠️ Unexpected error: {e}")
            return ""


In [35]:
speech_to_text(phrase_time_limit=1)

🎤 Listening Please speak...


### TTS

In [43]:
# ================== TTS ====================
import pyttsx3
import platform
import os

if platform.system() == "Windows":
    import pyttsx3
    engine = pyttsx3.init()
    def text_to_speech(text):
        engine.say(text)
        # engine.save_to_file(filename="output.wav", text=text)
        engine.runAndWait()
    def stop_tts():
        engine.stop()
else:
    # On non-Windows systems (Linux/macOS), use gTTS and mpg123
    # You will need to install gTTS and mpg123:
    # pip install gTTS
    # On Linux: sudo apt-get install mpg123
    # On macOS: brew install mpg123
    from gtts import gTTS
    def text_to_speech(text):
        tts = gTTS(text, lang='en')
        tts.save("temp_speech.mp3")
        os.system("mpg123 -q temp_speech.mp3")
        # os.remover("temp_speech.mp3")
    def stop_tts():
        pass # Not stoppable directly with mpg123

# text_to_speech(text="Hello My name is kamal jit singh. I am a data sciectiest and AI Engineer!")

### LLM Response

In [44]:
# ============= LLM Response ==========
from groq import Groq
import os
from dotenv import load_dotenv
load_dotenv()


GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# ================== LLM RESPONSE ==================
def generate_response(user_input: str):
    prompt = (
        "You are a helpful voice assistant. "
        "Respond clearly and concisely. "
        "Summarize long answers in 50–100 words. "
        "For factual queries, give direct answers. "
        f"User Query:\n{user_input}"
    )
    client = Groq(api_key=GROQ_API_KEY)
    chat_completion = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.1-8b-instant",
        temperature=0.4
    )
    return chat_completion.choices[0].message.content.strip()


### Voice detection

In [17]:
# voice detection fucntion.
# ======================= VAD Testing
import sounddevice as sd
import numpy as np

def voice_activity_detection(duration=2, samplerate=16000, threshold=0.6):
    """
    Detects if any significant sound is present (not just silence).
    Returns True if sound is detected, else False.
    """
    print("Recording... VAD is active.")
    audio = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype='float32')
    sd.wait()

    # Check the max absolute amplitude
    max_amplitude = np.max(np.abs(audio))
    print(f"Max amplitude detected: {max_amplitude:.4f}")
    if max_amplitude > threshold:
        print("Audio detected!")
        # sd.play(audio, samplerate)
        sd.wait()
        detected = True
    else:
        print("Audio not detected!")
        detected = False
    
    return detected
    
# print(voice_detection())

In [19]:
voice_activity_detection()

Recording... VAD is active.
Max amplitude detected: 0.9401
Audio detected!


True

In [ ]:
import threading


def speak_with_burge_in(stt_text):

    event = threading.Event()

    def play_tts():
        text_to_speech(text=stt_text)

    # create a thread for tts
    tts_thread = threading.Thread(target=play_tts)

    # start the thread
    tts_thread.start()

    while tts_thread.is_alive():
        

In [ ]:
import threading
import time


def speak_with_burge(stt_text: str):

    EXIT_COMMAND = ["stop", "exit", "stop listening", "goodby", "bye"]

    def play_tts():
        text_to_speech(text=stt_text)

    event = threading.Event()

    tts_thread = threading.Thread(target=play_tts)

    tts_thread.start()

    while tts_thread.is_alive():
        if voice_activity_detection():
            new_stt_text = speech_to_text(phrase_time_limit=3)
            
            if new_stt_text:
                if new_stt_text in EXIT_COMMAND:
                    text_to_speech(f"You said {new_stt_text}. Goodbye!")
                    stop_tts()
                    time.sleep(0.3)
                    break
                elif new_stt_text:
                    stop_tts()
                    time.sleep(0.2)
                    response = generate_response(user_input=new_stt_text)
                    text_to_speech(response)
                else: continue

                    


In [ ]:
def main():

    EXIT_COMMAND = ["stop", "exit", "stop listening", "goodby", "bye"]
    user_input = speech_to_text()

    while True:
        if user_input:
            if user_input in EXIT_COMMAND:
                text_to_speech(f"You said {user_input}. So Goodbye!")
                stop_tts()
                time.sleep(0.3)
                break
            else:
                response = generate_response(user_input=user_input)
                speak_with_burge_in(stt_text=response)
        else:
            speak_with_burge("Please speak loudly, I didn't get what you said.")


In [ ]:
import threading
import time

def barge_in(tts_text: str):
    """
    Runs TTS and voice detection in parallel.
    If the user interrupts with speech, stops TTS and returns the new speech text.
    Returns user's new speech (string) if detected, or None if no interruption.
    """

    barge_in_event = threading.Event()
    user_text = [None]

    def tts_worker():
        text_to_speech(tts_text)
        barge_in_event.set()

    def listen_worker():
        while not barge_in_event.is_set():
            detected = speech_to_text(phrase_time_limit=2)
            if detected:
                user_text[0] = detected.replace("[User]:", "").strip()
                stop_tts()
                barge_in_event.set()
                break
            time.sleep(0.1)

    tts_thread = threading.Thread(target=tts_worker)
    listen_thread = threading.Thread(target=listen_worker)
    tts_thread.start()
    listen_thread.start()
    tts_thread.join()
    listen_thread.join()

    return user_text[0]  # Returns new user speech if any, or None if none detected


## asyncio

In [21]:
import asyncio

async def barge_in(tts_text: str):
    """
    Natural conversation barge-in support using asyncio.
    - Plays TTS
    - Continuously listens in background
    - If user interrupts, TTS is stopped and new speech is returned
    """

    user_text = [None]
    stop_event = asyncio.Event()

    async def tts_worker():
        for sentence in tts_text.split(". "):  # chunk TTS
            if stop_event.is_set():
                break
            text_to_speech(sentence)  # blocking, but fast enough in chunks
            await asyncio.sleep(0.1)  # yield to event loop
        stop_event.set()

    async def listen_worker():
        while not stop_event.is_set():
            detected = speech_to_text()  # your STT, short phrase limit recommended
            if detected:
                user_text[0] = detected.strip()
                stop_tts()  # will only work on pyttsx3 (Windows)
                stop_event.set()
                break
            await asyncio.sleep(0.1)

    # Run TTS + listening concurrently
    await asyncio.gather(tts_worker(), listen_worker())

    return user_text[0]  # return user speech if interruption, else None


In [ ]:
async def conversation():
    while True:
        user_input = speech_to_text()
        if not user_input:
            continue

        if user_input.lower() in ["stop", "exit", "quit", "goodbye"]:
            text_to_speech("Goodbye! Have a nice day.")
            break

        response = generate_response(user_input)
        print(f"[Assistant]: {response}")

        # Speak with barge-in support
        interruption = await barge_in(response)

        if interruption:
            print(f"[User interrupted]: {interruption}")
            # handle new input immediately (loop continues)


## Threading

In [ ]:
import threading
import time

def barge_in(tts_text: str):
    """
    Speak text while simultaneously listening for user interruption.
    If the user speaks during TTS, stop TTS and return user speech.
    Returns: str (user's new speech) or None if no interruption.
    """

    stop_event = threading.Event()
    user_text = [None]

    def tts_worker():
        text_to_speech(tts_text, stop_event=stop_event)  
        stop_event.set()

    def listen_worker():
        while not stop_event.is_set():
            detected = speech_to_text(phrase_time_limit=2)
            if detected:
                user_text[0] = detected.replace("[User]:", "").strip()
                stop_event.set()
                stop_tts()  # immediately stop TTS if speaking
                break
            time.sleep(0.1)  # avoid busy loop

    # Start both workers
    tts_thread = threading.Thread(target=tts_worker, daemon=True)
    listen_thread = threading.Thread(target=listen_worker, daemon=True)

    tts_thread.start()
    listen_thread.start()

    tts_thread.join()
    listen_thread.join()

    return user_text[0]  # None if no interruption


### Main Function to run this app

In [ ]:
# ----------------------------
# Main Conversation Loop
# ----------------------------
def main():
    print("🤖 Voice Assistant started. Say 'exit' to quit.")

    while True:
        # Step 1: Listen to user
        user_input = speech_to_text()
        if not user_input:
            continue

        print(f"👤 User: {user_input}")

        if "exit" in user_input.lower():
            print("👋 Goodbye!")
            break

        # Step 2: Generate LLM response
        agent_reply = generate_response(user_input)
        print(f"🤖 Agent (planned reply): {agent_reply}")

        # Step 3: Speak response with barge-in support
        interrupted_text = barge_in(agent_reply)

        if interrupted_text:
            # User interrupted while agent was speaking
            print(f"👤 (Interruption): {interrupted_text}")
            followup_reply = generate_response(interrupted_text)
            barge_in(followup_reply)  # Agent responds again naturally


# ----------------------------
# Run the assistant
# ----------------------------
if __name__ == "__main__":
    main()

In [ ]:
# ============== STT =======================
import speech_recognition as sr


def speech_to_text(phrase_time_limit: int = None):
    speech = sr.Recognizer()
    print("🎤 Listening Please speak...")

    with sr.Microphone() as source:

        try:
            speech.adjust_for_ambient_noise(source)
            audio = speech.listen(source, phrase_time_limit=phrase_time_limit)
            user_input = speech.recognize_google(audio)
            return f"[User]: {user_input}"
        except sr.UnknownValueError:
            return None  # Return empty string if speech is unintelligible
        except sr.RequestError as e:
            print(f"Could not request results from Google Speech Recognition service; {e}")
            return None
        except sr.WaitTimeoutError:
            return None
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            return None
        

# ================== TTS ====================
import pyttsx3
import platform
import os

if platform.system() == "Windows":
    import pyttsx3
    engine = pyttsx3.init()
    def text_to_speech(text):
        engine.say(text)
        # engine.save_to_file(filename="output.wav", text=text)
        engine.runAndWait()
    def stop_tts():
        engine.stop()
else:
    # On non-Windows systems (Linux/macOS), use gTTS and mpg123
    # You will need to install gTTS and mpg123:
    # pip install gTTS
    # On Linux: sudo apt-get install mpg123
    # On macOS: brew install mpg123
    from gtts import gTTS
    def text_to_speech(text):
        tts = gTTS(text, lang='en')
        tts.save("temp_speech.mp3")
        os.system("mpg123 -q temp_speech.mp3")
        # os.remover("temp_speech.mp3")
    def stop_tts():
        pass # Not stoppable directly with mpg123

# text_to_speech(text="Hello My name is kamal jit singh. I am a data sciectiest and AI Engineer!")


# ============= LLM Response ==========
from groq import Groq
import os
from dotenv import load_dotenv
load_dotenv()


GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# ================== LLM RESPONSE ==================
def generate_response(user_input: str):
    prompt = (
        "You are a helpful voice assistant. "
        "Respond clearly and concisely. "
        "Summarize long answers in 50–100 words. "
        "For factual queries, give direct answers. "
        f"User Query:\n{user_input}"
    )
    client = Groq(api_key=GROQ_API_KEY)
    chat_completion = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.1-8b-instant",
        temperature=0.4
    )
    return chat_completion.choices[0].message.content.strip()


# voice detection fucntion.
# ======================= VAD Testing
import sounddevice as sd
import numpy as np

def voice_activity_detection(duration=2, samplerate=16000, threshold=0.6):
    """
    Detects if any significant sound is present (not just silence).
    Returns True if sound is detected, else False.
    """
    print("Recording... VAD is active.")
    audio = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype='float32')
    sd.wait()

    # Check the max absolute amplitude
    max_amplitude = np.max(np.abs(audio))
    print(f"Max amplitude detected: {max_amplitude:.4f}")
    if max_amplitude > threshold:
        print("Audio detected!")
        # sd.play(audio, samplerate)
        sd.wait()
        detected = True
    else:
        print("Audio not detected!")
        detected = False
    
    return detected
    
# print(voice_detection())

In [ ]:
import threading
import time

# Use your provided functions:
# - speech_to_text
# - text_to_speech
# - voice_detection
# - generate_response
# - stop_tts  (for Windows; does nothing on Linux/Mac per your logic)

def barge_in_voice_assistant():
    exit_words = ["stop", "exit", "quit"]
    last_input = "Hello! How can I assist you today?"
    stop_event = threading.Event()

    def tts_worker(text, stop_event):
        text_to_speech(text)
        stop_event.set()  # Signal done

    while True:
        stop_event.clear()
        tts_thread = threading.Thread(target=tts_worker, args=(last_input, stop_event))
        tts_thread.start()

        # Run detection loop in foreground so STT does not block within detection window
        detected_text = None
        while tts_thread.is_alive():  
            sound_text = voice_detection()  # listens for 3 seconds for voice
            if sound_text:
                detected_text = sound_text.replace("[User]:", "").strip()
                print(f"Detected user: {detected_text}")
                # Barge in! Stop the TTS and break the TTS loop
                stop_tts()
                break
            time.sleep(0.3)  # Slight pause to prevent busy looping

        tts_thread.join()  # Ensure TTS thread finishes or is stopped
        
        # If user said "stop", exit
        if detected_text and any(word in detected_text.lower() for word in exit_words):
            print("Assistant exiting as per user's request.")
            stop_tts()
            break

        # If no speech detected during TTS playback
        if not detected_text:
            reply_text = "I didn’t get what you said. Please speak loudly."
            print(reply_text)
            last_input = reply_text
            continue

        # If new, non-exit text—call LLM and start a new TTS loop
        reply_text = generate_response(detected_text)
        print(f"Assistant reply: {reply_text}")
        last_input = reply_text

# Run the assistant!
if __name__ == "__main__":
    barge_in_voice_assistant()


🎤 Listening Please speak...
🎤 Listening Please speak...
Detected user: hello
